In [0]:
from pyspark.sql import functions as F
from datetime import datetime

CATALOG = "intelligent_etl"
SCHEMA  = "default"
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {RUN_ID}")

Run ID: 20260513_132801


In [0]:
scored = spark.table(f"{CATALOG}.{SCHEMA}.silver_orders_scored")

clean_df      = scored.filter(F.col("is_anomaly") == False)
quarantine_df = scored.filter(F.col("is_anomaly") == True)

print(f"Clean rows:      {clean_df.count():,}")
print(f"Quarantine rows: {quarantine_df.count():,}")

Clean rows:      9,161
Quarantine rows: 718


In [0]:
# Gold clean — analytics ready
gold_clean = (
    clean_df
    .select(
        "order_id", "customer_id", "product_id", "product_name", "category",
        "quantity", "unit_price", "total_amount", "discount_pct",
        F.round(
            F.col("total_amount") * (1 - F.coalesce(F.col("discount_pct"), F.lit(0))), 2
        ).alias("net_revenue"),
        "order_date", "order_hour", "order_dow", "order_month", "order_year",
        "region", "payment_method", "status", "revenue_tier",
        "_ingestion_ts", "_run_id"
    )
    .withColumn("_gold_ts", F.current_timestamp())
)

(
    gold_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_year", "order_month")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_orders_clean")
)

print(f"Gold clean written: {spark.table(f'{CATALOG}.{SCHEMA}.gold_orders_clean').count():,} rows")

Gold clean written: 9,161 rows


In [0]:
# Gold quarantine — anomalies with full reason codes
gold_quarantine = (
    quarantine_df
    .select(
        "order_id", "customer_id", "product_id", "product_name", "category",
        "quantity", "unit_price", "total_amount",
        "order_date", "region", "payment_method", "status",
        "rule_flag", "zscore_flag", "if_flag", "if_score",
        "anomaly_reason",
        *[c for c in quarantine_df.columns if c.startswith("dq_")],
        "_ingestion_ts", "_run_id"
    )
    .withColumn("_quarantine_ts", F.current_timestamp())
)

(
    gold_quarantine
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_orders_quarantine")
)

print(f"Gold quarantine written: {spark.table(f'{CATALOG}.{SCHEMA}.gold_orders_quarantine').count():,} rows")

Gold quarantine written: 718 rows


In [0]:
clean      = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_clean")
quarantine = spark.table(f"{CATALOG}.{SCHEMA}.gold_orders_quarantine")

total      = clean.count() + quarantine.count()
anomalies  = quarantine.count()
rate       = anomalies / total * 100

print("="*50)
print("INTELLIGENT ETL PIPELINE — FINAL SUMMARY")
print("="*50)
print(f"  Bronze (raw):         9,879 rows")
print(f"  Silver (typed+DQ):    9,879 rows")
print(f"  Gold clean:           {clean.count():,} rows")
print(f"  Gold quarantine:      {quarantine.count():,} rows")
print(f"  Anomaly rate:         {rate:.1f}%")
print("="*50)
print("\nGold Clean — sample:")
clean.select("order_id","product_name","total_amount","net_revenue","revenue_tier").show(5)

print("\nGold Quarantine — sample with reasons:")
quarantine.select("order_id","anomaly_reason").show(5, truncate=False)

INTELLIGENT ETL PIPELINE — FINAL SUMMARY
  Bronze (raw):         9,879 rows
  Silver (typed+DQ):    9,879 rows
  Gold clean:           9,161 rows
  Gold quarantine:      718 rows
  Anomaly rate:         7.3%

Gold Clean — sample:
+--------------------+-------------+------------+-----------+------------+
|            order_id| product_name|total_amount|net_revenue|revenue_tier|
+--------------------+-------------+------------+-----------+------------+
|2189d919-9508-483...| Coffee Maker|    20067.24|    19786.3|      Medium|
|21dd44d0-b175-413...|   Headphones|    14647.14|    13548.6|      Medium|
|22336a0d-53e8-4f2...|        Novel|     1333.56|     1176.2|         Low|
|22a0c5f5-67d3-48d...|   Headphones|     4451.55|    3766.01|         Low|
|22e9944e-097c-42d...|Running Shoes|     9706.58|    9230.96|         Low|
+--------------------+-------------+------------+-----------+------------+
only showing top 5 rows

Gold Quarantine — sample with reasons:
+------------------------------

In [0]:
# List all tables we created
tables = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
tables.show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|       bronze_orders|      false|
| default|   gold_orders_clean|      false|
| default|gold_orders_quara...|      false|
| default|          orders_raw|      false|
| default|       silver_orders|      false|
| default|silver_orders_scored|      false|
+--------+--------------------+-----------+

